# Suite IDN — Identity: ownership and paths

A reference object's identity is its path: the route from a singleton to it through the relations that own it
(docs/OBJECT_IDENTITY.md). An adjacency marked `owner()` points back to the object's owner, through a relation whose
clauses make the route unambiguous; a key property may carry a naming pattern; a path is a nested value of
`Paths.OfRoot`, read off an object's ownership entries on demand. An object is a singleton, owned, or counted: a counted
object has no identity beyond the document that writes it, which numbers it.

In [ ]:
import json

from mbse.Schemas.Framework import Builders as B, Paths, Plain, Schemas as S, Validators
from mbse.Schemas.Framework.Errors import DecodeError, ReadOnlyError
from support import raises


def wts(store):
    """The directory of definitions, each owning its clauses, built in `store`: a singleton wts.Features owns each
    wts.Definition through wts.Defines, by a name its pattern gives, and a definition each wts.Clause through
    wts.HasClause."""
    Text = store.Schemas.OfStr.as_schema()
    Directory = store.Schemas.OfObject(lambda o: o.name("wts.Directory").ref(True).singleton("wts.Features"))
    Definition = store.Schemas.OfObject(lambda o: o.name("wts.Definition").ref(True).properties([
        lambda t: t.name("title").schema([Text]),
    ]))
    Clause = store.Schemas.OfObject(lambda o: o.name("wts.Clause").ref(True))
    Defines = store.Schemas.OfRelation(lambda r: (
        r
        .name("wts.Defines")
        .links([
            lambda t: t.name("directory"),
            lambda t: t.name("defn"),
        ])
        .properties([lambda t: t.name("name").schema([Text]).pattern(r"DEF-(\d+\d\d)")])
        .uniques([["directory", "name"], ["defn"]])
    ))
    HasClause = store.Schemas.OfRelation(lambda r: (
        r
        .name("wts.HasClause")
        .links([
            lambda t: t.name("defn"),
            lambda t: t.name("clause"),
        ])
        .properties([lambda t: t.name("name").schema([Text])])
        .uniques([["defn", "name"], ["clause"]])
    ))
    store.Schemas.OfObject(Directory, lambda o: o.adjacencies([lambda t: t.name("dir").relation([Defines]).me("directory")]))
    store.Schemas.OfObject(Definition, lambda o: o.adjacencies([
        lambda t: t.name("def").relation([Defines]).me("defn").owner(True),
        lambda t: t.name("clauses").relation([HasClause]).me("defn"),
    ]))
    store.Schemas.OfObject(Clause, lambda o: o.adjacencies([lambda t: t.relation([HasClause]).me("clause").owner(True)]))
    return Directory, Definition, Clause, Defines, HasClause


store = B.Store()
Text = store.Schemas.OfStr.as_schema()
Directory, Definition, Clause, Defines, HasClause = wts(store)

## IDN-01 · An adjacency marked `owner()`: its relation has two links, and clauses by which the owned object determines its owner and the owner and keys determine the owned object

In [ ]:
assert Definition.adjacencies["def"].owner is True and Directory.adjacencies["dir"].owner is None
assert [s.validate() for s in (Directory, Definition, Clause, Defines, HasClause)] == [[]] * 5


def owner_of(uniques, links=("up", "down")):
    """An object schema owned through a relation of `links` with `uniques`: its validation."""
    relation = store.Schemas.OfRelation(lambda r: r.links([lambda t, n=n: t.name(n) for n in links]).uniques(uniques))
    return store.Schemas.OfObject(lambda o: o.ref(True).adjacencies([
        lambda t: t.name("up").relation([relation]).me("down").owner(True),
    ])).validate()


assert owner_of([["up"], ["down"]]) == []  # keyless: the owner alone determines the owned object
assert owner_of([["down"]]) == ["adjacency 'up' is its object's owner: no unique clause has 'up' without 'down', so that "
                                "the owned object determines its owner"]
assert owner_of([["up"]]) == ["adjacency 'up' is its object's owner: no unique clause has 'down' without 'up', so that "
                              "the owner and the keys determine the owned object"]
assert owner_of([["up"], ["down"]], ("up", "down", "side")) == [
    "adjacency 'up' is its object's owner: its relation has two links, the owner's and the owned's"]
with raises(TypeError, match="property 'owner' holds bool, got str"):
    store.Schemas.OfObject(lambda o: o.adjacencies([lambda t: t.relation([Defines]).me("defn").owner("yes")]))

## IDN-02 · A key property's naming pattern: the values it gives and reads, and what validation refuses

In [ ]:
name = Defines.entry.properties["name"]
assert name.pattern == r"DEF-(\d+\d\d)" and name.name_for(7) == "DEF-007" and name.name_for(1234) == "DEF-1234"
assert (name.number_of("DEF-693"), name.number_of("DEF-69"), name.number_of("FEA-693")) == (693, None, None)
assert S.OfProperty.Data("k", Text).number_of("DEF-693") is None  # no pattern
bad = store.Schemas.OfRelation(lambda r: (
    r
    .links([lambda t: t.name("a"), lambda t: t.name("b")])
    .properties([
        lambda t: t.name("k").schema([Text]).pattern("DEF-.*"),
        lambda t: t.name("n").schema([store.Schemas.OfInt]).pattern(r"(\d+)"),
    ])
))
assert bad.validate() == [
    "property 'k': naming pattern 'DEF-.*' is not literal text around one group of digits, e.g. 'FEA-(\\d+\\d\\d)'",
    "property 'n': a naming pattern gives text: the property's schema is Schemas.OfStr"]

## IDN-03 · Every store holds the path schemas; the members of its own relations are owned

In [ ]:
# A path's steps nest, so a step holds itself, which validation refuses of a value schema (Decided, 72): an open question
# until 4f. That is each one's only problem.
cycle = "a value schema cannot hold itself: value object schemas cannot have cycles; reference object schemas can"
for name in ("Paths.OfRoot", "Paths.OfEntry", "Paths.OfTarget", "Paths.Next"):
    assert all(p.endswith(cycle) for p in store.registered(name).validate()) and store.framework(store.registered(name))
assert list(store.Paths.OfRoot.as_schema().properties) == ["singleton", "next"]
assert list(store.Paths.OfEntry.as_schema().properties) == ["i", "o", "properties", "next"]
assert [b.name for b in store.Paths.Next.as_schema().branches] == ["entry", "target"]
other = B.Store()
assert other.Paths.OfRoot.as_schema() is not store.Paths.OfRoot.as_schema()  # each store's own
assert "instances" not in store.Schemas.OfObject.as_schema().adjacencies  # a schema holds no instances
owning = [(a.relation.name, a.me) for a in store.Schemas.OfProperty.as_schema().anonymous if a.owner]
assert owning == [("Schemas.HasProperty", "property")]
assert all(a.owner for a in store.Schemas.OfObject.as_schema().anonymous if a.relation.name == "Schemas.Defines")

## IDN-04 · An object's path on demand: read off its ownership entries back to a singleton, a nested value; refused where it has no owner, or more than one

In [ ]:
features = store.Instances.wts.Features
assert features is store.singleton("wts.Features")
store.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.name("DEF-693").defn(lambda d: (
    d
    .title("Has 88 keys")
    .clauses(lambda c: c.insert(lambda h: h.name("C-1").clause(lambda x: x)))
)))))
definition = features.dir({"name": "DEF-693"})
clause = definition.clauses({"name": "C-1"})
to_plain = Plain.ToPlain(store)
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(definition)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "dir", "o": "def", "properties": {"name": "DEF-693"}, "next": {"target": {}}}}}
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(clause)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "dir", "o": "def", "properties": {"name": "DEF-693"}, "next": {"entry": {
        "i": "clauses", "o": "", "properties": {"name": "C-1"}, "next": {"target": {}}}}}}}  # an anonymous owner
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(features)) == {"singleton": "wts.Features",
                                                                         "next": {"target": {}}}
assert Paths.same(Paths.of(clause), Paths.of(clause)) and not Paths.same(Paths.of(clause), Paths.of(definition))
assert Paths.same(None, None) and not Paths.same(Paths.of(clause), None)
store.wts.Definition(definition, lambda d: d.clauses(lambda a: a.remove(lambda e: e.clause(clause))))
with raises(LookupError, match="a 'wts.Clause' has no owner: its path is not known"):
    Paths.of(clause)  # its owning entry removed
Twice = store.Schemas.OfObject(lambda o: o.name("wts.Twice").ref(True))
Holds = store.Schemas.OfRelation(lambda r: r.name("wts.Holds").links([lambda t: t.name("a"), lambda t: t.name("b")])
                                 .uniques([["a"], ["b"]]))
store.Schemas.OfObject(Twice, lambda o: o.adjacencies([
    lambda t: t.name("up").relation([Holds]).me("b").owner(True),
    lambda t: t.name("down").relation([Holds]).me("a"),
]))
with store._writing():  # within one write, objects are given their owners when it ends: here, each other
    first, second = store.wts.Twice(lambda t: t), store.wts.Twice(lambda t: t)
    store.wts.Twice(first, lambda t: t.up(lambda a: a.insert(lambda e: e.a(second))))
    store.wts.Twice(second, lambda t: t.up(lambda a: a.insert(lambda e: e.a(first))))
with raises(LookupError, match="an ownership cycle: no singleton owns the object"):
    Paths.of(first)
pair = [store.Schemas.OfRelation(lambda r: r.name(name).links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([["a"], ["b"]]))
        for name in ("wts.Lefts", "wts.Rights")]  # named: each end's adjacency links it
store.Schemas.OfObject(Directory, lambda o: o.adjacencies([lambda t: t.relation([pair[0]]).me("a"),
                                                           lambda t: t.name("rights").relation([pair[1]]).me("a")]))
Split = store.Schemas.OfObject(lambda o: o.name("wts.Split").ref(True).adjacencies([
    lambda t: t.name("left").relation([pair[0]]).me("b").owner(True),
    lambda t: t.name("right").relation([pair[1]]).me("b").owner(True),
]))
split = store.wts.Split(lambda s: s.left(lambda a: a.insert(lambda e: e.a(features))))
assert to_plain(store.Paths.OfRoot.as_schema(), Paths.of(split)) == {"singleton": "wts.Features", "next": {"entry": {
    "i": "", "o": "left", "properties": {}, "next": {"target": {}}}}}  # the owner's adjacency anonymous
store.wts.Split(split, lambda s: s.right(lambda a: a.insert(lambda e: e.a(features))))
with raises(LookupError, match="a 'wts.Split' has more than one owner: its path is not known"):
    Paths.of(split)
with raises(ReadOnlyError, match="'as_identity' is not a property or adjacency of 'wts.Twice'"):
    store.wts.Twice(first, lambda t: t.as_identity("twice-9"))  # a program writes no identity

## IDN-05 · An object built without an owner is given the one its schema allows: a singleton's, its keys from their naming patterns; else refused. Removing its owning entry deletes it, and what it owns

In [ ]:
store.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.defn(lambda d: d.title("nameless")))))  # no key
made = store.wts.Definition(lambda d: d.title("made alone"))  # its owner the one singleton that can be, Features
assert features.dir({"name": "DEF-694"}) is made  # its key the next its pattern gives, past every number held
assert store.wts.Definition(lambda d: d) is features.dir({"name": "DEF-695"})
with raises(ValueError, match="a 'wts.Clause' needs an owner through 'wts.HasClause': give it one, since no singleton can be"):
    store.wts.Clause(lambda c: c)  # a definition owns it, and none is a singleton
with raises(ValueError, match="a 'wts.Split' needs an owner through 'left', 'right': give it one, since more than one "
                              "singleton can be"):
    store.wts.Split(lambda s: s)
Notes = store.Schemas.OfRelation(lambda r: (
    r
    .name("wts.Notes")
    .links([lambda t: t.name("directory"), lambda t: t.name("note")])
    .properties([lambda t: t.name("name").schema([Text])])  # a key with no naming pattern
    .uniques([["directory", "name"], ["note"]])
))
store.Schemas.OfObject(Directory, lambda o: o.adjacencies([lambda t: t.name("notes").relation([Notes]).me("directory")]))
store.Schemas.OfObject(lambda o: o.name("wts.Note").ref(True).adjacencies([
    lambda t: t.relation([Notes]).me("note").owner(True),
]))
with raises(ValueError, match="a 'wts.Note' needs an owner: the key 'name' of 'wts.Notes' has no naming pattern to give "
                              "it a value"):
    store.wts.Note(lambda n: n)
store.wts.Directory(features, lambda f: f.notes(lambda a: a.insert(lambda e: e.name("first").note(lambda n: n))))
assert features.notes({"name": "first"}).as_schema() is store.registered("wts.Note")

# Removing an owning entry deletes the owned object: every entry it is in goes, and so, in turn, what it owned.
gone = store.wts.Definition(lambda d: d.title("gone").clauses(lambda c: c.insert(lambda h: h.name("C-1").clause(lambda x: x))))
its = gone.clauses({"name": "C-1"})
store.wts.Directory(features, lambda f: f.dir(lambda a: a.remove(lambda e: e.defn(gone))))
assert len(gone.clauses) == 0 and all(e.defn is not gone for e in features.dir)
with raises(LookupError, match="a 'wts.Clause' has no owner: its path is not known"):
    Paths.of(its)  # deleted with the definition that owned it

# Nothing is owned by a counted object: an object neither a singleton nor owned.
Loose = store.Schemas.OfObject(lambda o: o.name("wts.Loose").ref(True))
Keeps = store.Schemas.OfRelation(lambda r: r.name("wts.Keeps").links([lambda t: t.name("keeper"), lambda t: t.name("kept")])
                                 .uniques([["keeper"], ["kept"]]))
store.Schemas.OfObject(Loose, lambda o: o.adjacencies([lambda t: t.name("keeps").relation([Keeps]).me("keeper")]))
store.Schemas.OfObject(lambda o: o.name("wts.Kept").ref(True).adjacencies([lambda t: t.relation([Keeps]).me("kept").owner(True)]))
with raises(ValueError, match="a 'wts.Kept' cannot be owned by a counted 'wts.Loose': nothing is owned by a counted object"):
    store.wts.Loose(lambda l: l.keeps(lambda a: a.insert(lambda e: e.kept(lambda k: k))))
with raises(ValueError, match="nothing is owned by a counted object"), store._writing():  # refused within another write
    store.wts.Loose(lambda l: l.keeps(lambda a: a.insert(lambda e: e.kept(lambda k: k))))
loose = store.wts.Loose(lambda l: l)
assert loose.as_schema() is Loose  # a refused write leaves nothing pending for the next
assert Validators.Validate(store)(Loose, loose) == [  # and validation refuses the schemas that would allow it
    "schema 'wts.Loose': adjacency 'keeps' would own a 'wts.Kept', through its adjacency marked owner(): nothing is owned "
    "by a counted object, which a 'wts.Loose' is"]
Stray = store.Schemas.OfObject(lambda o: o.name("wts.Stray").ref(True).adjacencies([lambda t: t.relation([Keeps]).me("keeper")]))
assert Validators.Validate(store)(Stray, store.wts.Stray(lambda s: s)) == [  # an anonymous adjacency, by its relation
    "schema 'wts.Stray': adjacency 'wts.Keeps' would own a 'wts.Kept', through its adjacency marked owner(): nothing is "
    "owned by a counted object, which a 'wts.Stray' is"]
assert Validators.Validate(store)(Directory, features) == []  # a singleton owns

## IDN-06 · Lookup by keys: on any adjacency; `store.Instances` finds only singletons, and a counted object has no path

In [ ]:
Free = store.Schemas.OfObject(lambda o: o.name("wts.Free").ref(True))
free = store.wts.Free(lambda f: f)  # no owner() adjacency, and not a singleton: counted
with raises(LookupError, match="a 'wts.Free' is counted: it has no path"):
    Paths.of(free)
assert repr(store.Instances.wts) == "<Instances.wts>"
with raises(LookupError, match="no singleton named 'wts.Free': an owned object is found through its owners' adjacencies, "
                               "a counted one only through what links it"):
    store.Instances.wts.Free({"name": "free-1"})
with raises(LookupError, match="no singleton named 'wts.Definition'"):
    store.Instances.wts.Definition({"name": "DEF-693"})  # owned through Defines: found as features.dir({"name": ...})
with raises(AttributeError, match="_hidden"):
    store.Instances._hidden
with raises(LookupError, match="no entry of 'dir' has the keys {'name': 'DEF-999'}"):
    features.dir({"name": "DEF-999"})
with raises(KeyError, match="'nope' is not a property of 'wts.Defines'"):
    features.dir({"nope": 1})
Trio = store.Schemas.OfRelation(lambda r: r.name("wts.Trio").links([lambda t, n=n: t.name(n) for n in ("a", "b", "c")]))
store.Schemas.OfObject(Directory, lambda o: o.adjacencies([lambda t: t.name("trio").relation([Trio]).me("a")]))
with raises(TypeError, match="a lookup through 'trio' needs a relation of two links"):
    features.trio({})

## IDN-07 · Owned objects in snapshots and logs: listed at their paths, read back owners first, each owned by the entry its path's last step gives

In [ ]:
def defined(store):
    """wts.Features, defining DEF-001, which holds the clause C-1."""
    features = store.Instances.wts.Features
    store.wts.Directory(features, lambda f: f.dir(lambda a: a.insert(lambda e: e.name("DEF-001").defn(lambda d: (
        d
        .title("Keys")
        .clauses(lambda c: c.insert(lambda h: h.name("C-1").clause(lambda x: x)))
    )))))
    return features


source, target = B.Store(), B.Store()
wts(source)
wts(target)
features = defined(source)
graph = Plain.ToPlain(source).Reachable(source.registered("wts.Directory"), features)
assert [o["schema"] for o in graph["objects"]] == ["wts.Directory", "wts.Definition", "wts.Clause"]
assert graph["objects"][1]["path"] == {"singleton": "wts.Features", "next": {"entry": {
    "i": "dir", "o": "def", "properties": {"name": "DEF-001"}, "next": {"target": {}}}}}
back = Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), graph)  # owners first; each owned by its path
assert back is target.Instances.wts.Features and str(back.dir({"name": "DEF-001"}).title) == "Keys"
assert Plain.ToPlain(target).Reachable(target.registered("wts.Directory"), back) == graph
assert Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), graph) is back and len(back.dir) == 1  # an update
clause = features.dir({"name": "DEF-001"}).clauses({"name": "C-1"})
with raises(DecodeError, match="$.objects: an object's owner is not in the snapshot"):
    Plain.FromPlain(target).OfObject(target.registered("wts.Clause"), Plain.ToPlain(source).OfObject(source.registered("wts.Clause"), clause))
renamed = json.loads(json.dumps(graph).replace('"o": "def"', '"o": "nope"'))
with raises(DecodeError, match="'wts.Definition' has no adjacency 'nope' marked owner() for a path to go through"):
    Plain.FromPlain(target).Reachable(target.registered("wts.Directory"), renamed)

# A log writes them with their paths, and reads them back where their paths place them.
log = Plain.ToPlain(source).Snapshot()
third = B.Store()
Plain.FromPlain(third).Snapshot(log)
assert Plain.ToPlain(third).Snapshot() == log and Plain.FromPlain(third).Snapshot(log) == []  # again: each exists
assert third.Instances.wts.Features.dir({"name": "DEF-001"}).clauses({"name": "C-1"}).as_schema() is third.registered("wts.Clause")
at = graph["objects"][1]["path"]
for definition, says in (
    ({"path": at}, "a data object's definition names its schema"),
    ({"path": at, "schema": "Paths.OfTarget"}, "'Paths.OfTarget' is not a reference object schema"),
    ({"path": json.loads(json.dumps(at).replace("DEF-001", "DEF-999")), "schema": "wts.Definition"}, None),
    ({"path": {"singleton": "wts.Features", "next": {"entry": {"i": "dir", "o": "def", "properties": {"name": "DEF-998"},
      "next": {"entry": {"i": "clauses", "o": "", "properties": {"name": "C-1"}, "next": {"target": {}}}}}}},
      "schema": "wts.Clause"}, "the object's owner is not in the store"),
):
    if says is None:
        Plain.FromPlain(third).Snapshot([{"exists": {"definition": definition}}])  # a new definition, owned as its path says
        assert third.Instances.wts.Features.dir({"name": "DEF-999"}).as_schema() is third.registered("wts.Definition")
        continue
    with raises(DecodeError, match=says):
        Plain.FromPlain(third).Snapshot([{"exists": {"definition": definition}}])
another = json.loads(json.dumps(graph).replace("DEF-001", "DEF-002"))  # the store's features hold DEF-001 and DEF-999
Directory3 = third.registered("wts.Directory")
assert Plain.FromPlain(third).Reachable(Directory3, another) is third.Instances.wts.Features
assert [str(e.name) for e in third.Instances.wts.Features.dir] == ["DEF-001", "DEF-999", "DEF-002"]  # a new one, owned
with raises(DecodeError, match="$.objects[0].n: a 'wts.Definition' is owned: it is called by its path, not a number"):
    Plain.FromPlain(third).OfObject(third.registered("wts.Definition"), {"root": {"n": 0}, "objects": [
        {"n": 0, "schema": "wts.Definition", "properties": {}}]})
with raises(DecodeError, match="$.objects[0].path: a 'wts.Definition' is not a singleton: its path goes through what owns it"):
    Plain.FromPlain(third).OfObject(third.registered("wts.Definition"), {"root": {"singleton": "wts.Features", "next": {
        "target": {}}}, "objects": [{"path": {"singleton": "wts.Features", "next": {"target": {}}}, "schema": "wts.Definition",
                                     "properties": {}}]})